# 04_04 — Validación integral del pipeline ETL

**Proyecto de Big Data · Fase 04: Limpieza, transformación e integración de datos**  
**Fuentes:** NYC Taxi Trip Records y NOAA (estación Central Park) · **Periodo:** 2024  
**Tecnología:** PySpark 4.1.3, Parquet y Docker

## Objetivo

Verificar que los conjuntos de datos persistidos en la **capa Silver** mantengan la estructura, los volúmenes, los indicadores de calidad y los valores meteorológicos esperados después de la integración. Esta etapa **audita los resultados almacenados**; no vuelve a limpiar los archivos de origen ni a generar el dataset integrado.

### Flujo auditado

```text
Taxi Silver (41,169,664 viajes) ──┐
                                 ├── LEFT JOIN por fecha ──> Taxi + NOAA Silver
NOAA Silver (366 días) ──────────┘                         (41,169,664 viajes)
                                              │
                                              └── Controles de integridad y calidad
```

### Criterios de aceptación

| Dimensión | Comprobación principal |
|---|---|
| Estructura | Columnas críticas presentes y claves temporales tipadas |
| Volumen | Igualdad mensual entre Taxi Silver y Taxi + NOAA |
| Cobertura | Ningún viaje sin correspondencia meteorológica |
| Consistencia | Variables NOAA correctas para cada fecha |
| Trazabilidad | Indicadores heredados sin cambios en sus conteos mensuales |

> **Alcance:** los controles verifican principalmente la consistencia **entre datasets Silver**, no la exactitud física de todos los registros originales ni la identidad fila por fila de cada viaje.


## 1. Entorno de ejecución

Se inicializa una sesión local de Spark con cuatro hilos y 24 particiones de *shuffle*. Al igual que en los notebooks anteriores, la raíz del repositorio se detecta a partir de las carpetas `data/` y `notebooks/`.

Esta sección define el entorno; las acciones que recorren los Parquet se realizan en las auditorías siguientes. La configuración es compatible con el flujo local utilizado para el proyecto.


In [1]:
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# --------------------------------------------------
# 1. Localizar el proyecto
# --------------------------------------------------

def find_project_root(start: Path) -> Path:
    """Localiza la raíz del repositorio."""

    for candidate in [start, *start.parents]:
        if (
            (candidate / "data").is_dir()
            and (candidate / "notebooks").is_dir()
        ):
            return candidate

    raise FileNotFoundError(
        "No se encontró la raíz del proyecto."
    )


PROJECT_ROOT = find_project_root(Path.cwd())

# --------------------------------------------------
# 2. Inicializar Spark
# --------------------------------------------------

spark = (
    SparkSession.builder
    .appName("04_04_pipeline_validation")
    .master("local[4]")
    .config("spark.sql.shuffle.partitions", "24")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

print("Proyecto:", PROJECT_ROOT)
print("Spark:", spark.version)

Proyecto: c:\Users\derec\Desktop\Proyectos\BigData_project
Spark: 4.1.3


## 2. Lectura de los datasets Silver

Cargamos los **tres resultados persistidos** del pipeline, sin volver a ejecutar la limpieza ni el JOIN. De este modo también verificamos que los Parquet pueden leerse después de su publicación.

| Dataset | Ruta relativa | Función en la auditoría |
|---|---|---|
| Taxi Silver | `data/processed/taxi/silver/` | Referencia de registros e indicadores |
| NOAA Silver | `data/processed/weather/silver/` | Referencia de variables meteorológicas |
| Taxi + NOAA Silver | `data/processed/taxi_weather/silver/` | Resultado integrado que se evalúa |

**Resultado observado:** Taxi Silver tiene **43 columnas**, NOAA Silver **15** y el integrado **59**. El aumento obedece a las variables NOAA incorporadas y a dos indicadores de integración, mientras `weather_date` se elimina tras el JOIN.


In [2]:
# Rutas de las capas Silver
DATASETS = {
    "taxi": (
        PROJECT_ROOT / "data/processed/taxi/silver"
    ),
    "weather": (
        PROJECT_ROOT / "data/processed/weather/silver"
    ),
    "integrated": (
        PROJECT_ROOT / "data/processed/taxi_weather/silver"
    )
}

# Verificar existencia de las rutas
for name, path in DATASETS.items():
    assert path.exists(), (
        f"No existe el dataset {name}: {path}"
    )

# Lectura de datasets
df_taxi = spark.read.parquet(str(DATASETS["taxi"]))

df_weather = spark.read.parquet(
    str(DATASETS["weather"])
)

df_integrated = spark.read.parquet(
    str(DATASETS["integrated"])
)

# Resumen estructural inicial
for name, df in [
    ("Taxi Silver", df_taxi),
    ("NOAA Silver", df_weather),
    ("Taxi + NOAA", df_integrated)
]:
    print(
        f"{name}: {len(df.columns)} columnas"
    )

Taxi Silver: 43 columnas
NOAA Silver: 15 columnas
Taxi + NOAA: 59 columnas


## 3. Validación estructural

Se inspecciona el **contrato mínimo de esquema**: presencia de las columnas necesarias, ausencia de nombres repetidos y tipo `date` en las claves temporales (`pickup_date` y `weather_date`).

La verificación se concentra en las variables esenciales, en lugar de exigir una lista exhaustiva de las 59 columnas. Esto facilita detectar rupturas relevantes del pipeline sin acoplar el notebook a columnas auxiliares que podrían cambiar.

**Criterio de aprobación:** ningún campo obligatorio faltante, sin nombres duplicados y claves de fecha compatibles.


In [3]:
from pyspark.sql.types import DateType


# --------------------------------------------------
# 1. Definir columnas obligatorias
# --------------------------------------------------

REQUIRED_COLUMNS = {

    "taxi": [
        "pickup_date",
        "pickup_month",
        "source_month",
        "trip_distance",
        "duration_seconds",
        "is_flex_fare",
        "flag_invalid_duration",
        "is_mobility_core"
    ],

    "weather": [
        "weather_date",
        "awnd_mps",
        "prcp_mm",
        "snow_mm",
        "tmax_c",
        "tmin_c",
        "flag_missing_awnd"
    ],

    "integrated": [
        "pickup_date",
        "pickup_month",
        "trip_distance",
        "prcp_mm",
        "awnd_mps",
        "flag_weather_not_matched",
        "flag_weather_missing_awnd"
    ]
}

dataframes = {
    "taxi": df_taxi,
    "weather": df_weather,
    "integrated": df_integrated
}

# --------------------------------------------------
# 2. Validar columnas
# --------------------------------------------------

for name, df in dataframes.items():

    missing_columns = (
        set(REQUIRED_COLUMNS[name])
        - set(df.columns)
    )

    assert not missing_columns, (
        f"{name}: faltan columnas {missing_columns}"
    )

    assert len(df.columns) == len(set(df.columns)), (
        f"{name}: existen columnas duplicadas"
    )

    print(f"{name}: esquema de columnas correcto.")

# --------------------------------------------------
# 3. Validar tipos de claves temporales
# --------------------------------------------------

assert isinstance(
    df_taxi.schema["pickup_date"].dataType,
    DateType
)

assert isinstance(
    df_weather.schema["weather_date"].dataType,
    DateType
)

assert isinstance(
    df_integrated.schema["pickup_date"].dataType,
    DateType
)

print("Validación estructural completada.")

taxi: esquema de columnas correcto.
weather: esquema de columnas correcto.
integrated: esquema de columnas correcto.
Validación estructural completada.


**Interpretación.** Los tres esquemas cumplen los controles establecidos. Esto valida la estructura necesaria para el análisis, pero no sustituye una verificación completa de todos los tipos de datos, dominios y restricciones de negocio.


## 4. Conciliación mensual de registros

La unión con NOAA tiene cardinalidad esperada **muchos-a-uno**: cada viaje debe encontrar como máximo una observación meteorológica de su fecha. Para detectar pérdidas o multiplicaciones de registros, se comparan los volúmenes de **cada mes de inicio** (`pickup_month`) entre Taxi Silver y el integrado.

Para cada mes $m$, se calcula:

$$
\Delta_m = N_m^{\mathrm{integrado}} - N_m^{\mathrm{taxi}}
$$

**Criterio de aprobación:** $\Delta_m = 0$ en los doce meses y ausencia de viajes con `flag_weather_not_matched = True`.

> `pickup_month` corresponde al mes **real de inicio del viaje**; no debe confundirse con `source_month`, que identifica el archivo original y puede diferir en viajes cercanos al cambio de mes.


In [4]:
# --------------------------------------------------
# 1. Conteos mensuales de Taxi Silver
# --------------------------------------------------

df_taxi_monthly = (
    df_taxi
    .groupBy("pickup_month")
    .agg(
        F.count("*").alias("taxi_records")
    )
)

# --------------------------------------------------
# 2. Conteos mensuales del dataset integrado
# --------------------------------------------------

df_integrated_monthly = (
    df_integrated
    .groupBy("pickup_month")
    .agg(
        F.count("*").alias("integrated_records"),

        # Fechas sin correspondencia NOAA
        F.sum(
            F.when(
                F.col("flag_weather_not_matched"), 1
            ).otherwise(0)
        ).alias("unmatched_weather"),

        # Viajes asociados con AWND faltante
        F.sum(
            F.when(
                F.col("flag_weather_missing_awnd"), 1
            ).otherwise(0)
        ).alias("missing_awnd_trips")
    )
)

# --------------------------------------------------
# 3. Conciliar los registros mensuales
# --------------------------------------------------

df_reconciliation = (
    df_taxi_monthly
    .join(
        df_integrated_monthly,
        on="pickup_month",
        how="full_outer"
    )
    .withColumn(
        "difference",
        F.col("integrated_records")
        - F.col("taxi_records")
    )
    .orderBy("pickup_month")
)

df_reconciliation.show(12, truncate=False)

+------------+------------+------------------+-----------------+------------------+----------+
|pickup_month|taxi_records|integrated_records|unmatched_weather|missing_awnd_trips|difference|
+------------+------------+------------------+-----------------+------------------+----------+
|2024-01     |2964617     |2964617           |0                |215635            |0         |
|2024-02     |3007533     |3007533           |0                |0                 |0         |
|2024-03     |3582611     |3582611           |0                |0                 |0         |
|2024-04     |3514295     |3514295           |0                |0                 |0         |
|2024-05     |3723843     |3723843           |0                |0                 |0         |
|2024-06     |3539170     |3539170           |0                |212491            |0         |
|2024-07     |3076876     |3076876           |0                |0                 |0         |
|2024-08     |2979192     |2979192           |0   

### 4.1 Comprobaciones automáticas

Se recuperan **únicamente doce filas** de la tabla mensual mediante `collect()`, una cantidad segura para el controlador. Las aserciones verifican que estén representados todos los meses, que no haya diferencias ni viajes sin NOAA y que coincidan los totales globales.


In [5]:
# --------------------------------------------------
# 1. Recuperar los doce resultados mensuales
# --------------------------------------------------

monthly_results = df_reconciliation.collect()

# --------------------------------------------------
# 2. Validar cada mes
# --------------------------------------------------

assert len(monthly_results) == 12

for row in monthly_results:

    month = row["pickup_month"]

    assert row["taxi_records"] is not None, (
        f"Faltan registros originales: {month}"
    )

    assert row["integrated_records"] is not None, (
        f"Faltan registros integrados: {month}"
    )

    assert row["difference"] == 0, (
        f"Diferencia de registros en {month}"
    )

    assert row["unmatched_weather"] == 0, (
        f"Viajes sin información NOAA en {month}"
    )

# --------------------------------------------------
# 3. Verificar resultados globales
# --------------------------------------------------

total_taxi = sum(
    row["taxi_records"]
    for row in monthly_results
)

total_integrated = sum(
    row["integrated_records"]
    for row in monthly_results
)

total_missing_awnd = sum(
    row["missing_awnd_trips"]
    for row in monthly_results
)

assert total_taxi == 41_169_664
assert total_integrated == 41_169_664
assert total_missing_awnd == 428_126

print("Conciliación mensual correcta.")
print(f"Taxi Silver: {total_taxi:,}")
print(f"Taxi + NOAA: {total_integrated:,}")
print(f"Viajes con AWND faltante: {total_missing_awnd:,}")

Conciliación mensual correcta.
Taxi Silver: 41,169,664
Taxi + NOAA: 41,169,664
Viajes con AWND faltante: 428,126


### 4.2 Interpretación de la conciliación

Los doce meses registran **diferencia cero**. Tanto Taxi Silver como Taxi + NOAA contienen **41,169,664 viajes**, sin fechas de inicio sin correspondencia meteorológica.

El indicador de viento faltante afecta a **428,126 viajes (aprox. 1.04%)**, concentrados en dos meses:

| Mes | Viajes con `AWND` faltante |
|---|---:|
| Enero | 215,635 |
| Junio | 212,491 |
| **Total** | **428,126** |

Este resultado es coherente con las cuatro fechas sin `AWND` identificadas durante la preparación de NOAA. Un valor meteorológico faltante **no equivale** a una fecha sin correspondencia: las otras variables de esos días siguen disponibles.

La igualdad de conteos es una condición **necesaria**, pero por sí sola no demuestra que los valores meteorológicos se hayan asociado correctamente. El siguiente control verifica ese aspecto.


## 5. Consistencia meteorológica por fecha

Para verificar el contenido, se agrupan los viajes por `pickup_date` y se obtienen, para cada variable meteorológica, su mínimo, máximo y número de valores no nulos. Luego se compara esa tabla de **366 fechas** con NOAA Silver mediante un pequeño `broadcast`.

Las cinco variables auditadas son:

- `awnd_mps`: velocidad media del viento (m/s).
- `prcp_mm`: precipitación (mm).
- `snow_mm`: nieve (mm).
- `tmax_c` y `tmin_c`: temperaturas máxima y mínima (°C).

Para una fecha con valor observado, todos sus viajes deben compartir **ese mismo valor**. Si NOAA presenta `NULL`, el conteo no nulo debe ser cero. Se utiliza `eqNullSafe()` para comparar explícitamente los valores faltantes, en lugar de tratar `NULL = NULL` como verdadero.

**Criterio de aprobación:** 366 fechas consistentes, sin viajes desconectados de NOAA y con conservación del indicador de viento faltante.


In [6]:
# --------------------------------------------------
# 1. Variables meteorológicas a validar
# --------------------------------------------------

WEATHER_VARIABLES = [
    "awnd_mps",
    "prcp_mm",
    "snow_mm",
    "tmax_c",
    "tmin_c"
]

# --------------------------------------------------
# 2. Agregar viajes por fecha
# --------------------------------------------------

aggregations = [
    F.count("*").alias("trip_count"),

    F.sum(
        F.col("flag_weather_not_matched").cast("long")
    ).alias("unmatched_trips"),

    F.sum(
        F.col("flag_weather_missing_awnd").cast("long")
    ).alias("missing_awnd_trips")
]

for variable in WEATHER_VARIABLES:

    aggregations.extend([
        F.min(variable).alias(f"min_{variable}"),
        F.max(variable).alias(f"max_{variable}"),
        F.count(variable).alias(f"count_{variable}")
    ])

df_daily_audit = (
    df_integrated
    .groupBy("pickup_date")
    .agg(*aggregations)
)

# --------------------------------------------------
# 3. Preparar referencia meteorológica
# --------------------------------------------------

df_weather_reference = df_weather.select(
    "weather_date",
    *[
        F.col(variable).alias(f"expected_{variable}")
        for variable in WEATHER_VARIABLES
    ]
)

# Solo se integran 366 registros agregados.
df_daily_audit = df_daily_audit.join(
    F.broadcast(df_weather_reference),
    df_daily_audit["pickup_date"]
    == df_weather_reference["weather_date"],
    "left"
)

# --------------------------------------------------
# 4. Verificar consistencia por fecha
# --------------------------------------------------

condition = (
    F.col("weather_date").isNotNull()
    & (F.col("unmatched_trips") == 0)
)

for variable in WEATHER_VARIABLES:

    expected = F.col(f"expected_{variable}")

    # Si NOAA tiene NULL, todos los viajes de ese día
    # deben mantener NULL en la variable.
    expected_count = F.when(
        expected.isNull(),
        F.lit(0)
    ).otherwise(F.col("trip_count"))

    condition = (
        condition
        & F.col(f"min_{variable}").eqNullSafe(expected)
        & F.col(f"max_{variable}").eqNullSafe(expected)
        & (
            F.col(f"count_{variable}")
            == expected_count
        )
    )

# Comprobar también el indicador de viento faltante.
condition = condition & (
    F.col("missing_awnd_trips")
    == F.when(
        F.col("expected_awnd_mps").isNull(),
        F.col("trip_count")
    ).otherwise(F.lit(0))
)

df_daily_audit = df_daily_audit.withColumn(
    "weather_consistent",
    condition
)

# --------------------------------------------------
# 5. Resultado de la auditoría
# --------------------------------------------------

weather_audit = df_daily_audit.agg(
    F.count("*").alias("dates_checked"),

    F.sum(
        F.when(
            ~F.col("weather_consistent"), 1
        ).otherwise(0)
    ).alias("dates_failed"),

    F.sum("trip_count").alias("trips_checked")
).first()

print("Fechas evaluadas:", weather_audit["dates_checked"])
print("Fechas inconsistentes:", weather_audit["dates_failed"])
print("Viajes evaluados:", weather_audit["trips_checked"])

assert weather_audit["dates_checked"] == 366
assert weather_audit["dates_failed"] == 0
assert weather_audit["trips_checked"] == 41_169_664

print("Consistencia meteorológica validada.")

Fechas evaluadas: 366
Fechas inconsistentes: 0
Viajes evaluados: 41169664
Consistencia meteorológica validada.


**Resultado.** Se auditaron **366 fechas y 41,169,664 viajes**, con **cero fechas inconsistentes**. Este control respalda la correcta asignación diaria de las variables meteorológicas, incluyendo los `NULL` de viento.

La validación se realiza a la **granularidad diaria**; no verifica condiciones meteorológicas por hora ni diferencias entre zonas de Nueva York.


## 6. Conservación de indicadores heredados

Los indicadores definidos durante la limpieza de NYC Taxi deben mantenerse iguales después del enriquecimiento meteorológico. Se comparan sus **conteos mensuales** en Taxi Silver y Taxi + NOAA.

Indicadores incluidos:

| Indicador | Interpretación |
|---|---|
| `flag_invalid_duration` | Duración nula o no positiva |
| `flag_zero_distance` | Distancia registrada igual a cero |
| `flag_negative_fare` | Tarifa base negativa |
| `flag_negative_total` | Importe total negativo |
| `is_mobility_core` | Viaje que supera las reglas operativas de selección de movilidad |

Este procedimiento comprueba que las **distribuciones mensuales** de los cinco indicadores no cambiaron; no prueba una correspondencia registro-a-registro, ya que la fuente no dispone de un identificador único de viaje utilizado en esta auditoría.


In [7]:
# Indicadores heredados de Taxi Silver
QUALITY_FLAGS = [
    "flag_invalid_duration",
    "flag_zero_distance",
    "flag_negative_fare",
    "flag_negative_total",
    "is_mobility_core"
]


def quality_summary_by_month(df, suffix):
    """
    Resume los indicadores de calidad por mes.

    Cuenta registros y observaciones donde cada
    indicador booleano es verdadero.
    """

    expressions = [
        F.count("*").alias(f"records_{suffix}")
    ]

    for flag in QUALITY_FLAGS:
        expressions.append(
            F.sum(
                F.when(F.col(flag), 1).otherwise(0)
            ).alias(f"{flag}_{suffix}")
        )

    return (
        df.groupBy("pickup_month")
        .agg(*expressions)
    )


# Agregar cada dataset
df_taxi_quality = quality_summary_by_month(
    df_taxi, "taxi"
)

df_integrated_quality = quality_summary_by_month(
    df_integrated, "integrated"
)

# Comparar resultados mensuales
df_quality_audit = df_taxi_quality.join(
    df_integrated_quality,
    on="pickup_month",
    how="full_outer"
)

quality_results = df_quality_audit.collect()

assert len(quality_results) == 12

for row in quality_results:

    assert (
        row["records_taxi"]
        == row["records_integrated"]
    ), f"Diferencia de registros en {row['pickup_month']}"

    for flag in QUALITY_FLAGS:

        assert (
            row[f"{flag}_taxi"]
            == row[f"{flag}_integrated"]
        ), (
            f"Inconsistencia en {flag}, "
            f"mes {row['pickup_month']}"
        )

print("Indicadores de calidad conservados.")
print("Meses evaluados:", len(quality_results))
print("Indicadores evaluados:", len(QUALITY_FLAGS))

Indicadores de calidad conservados.
Meses evaluados: 12
Indicadores evaluados: 5


**Resultado.** Los **cinco indicadores** coinciden en los **doce meses**. La integración no presenta alteraciones detectables en sus conteos mensuales, lo que refuerza la trazabilidad de las reglas de limpieza.


## 7. Reporte integral de calidad

El reporte reúne la evidencia obtenida en las validaciones anteriores y resume siete controles de aceptación. La tabla utiliza los estados `PASS` una vez que las verificaciones previas han terminado sin errores; **los estados no se calculan mediante una evaluación independiente adicional**.

Se consideran satisfactorias las dimensiones de estructura, volumen, cobertura, contenido meteorológico y conservación de indicadores.


In [8]:
# Construir reporte solamente después de ejecutar
# correctamente las validaciones anteriores.

report = [
    (
        "Integridad estructural",
        "PASS",
        "Esquemas y columnas requeridas correctos"
    ),
    (
        "Conciliación mensual",
        "PASS",
        "12 meses sin diferencias de registros"
    ),
    (
        "Conservación de viajes",
        "PASS",
        f"{total_integrated:,} viajes conservados"
    ),
    (
        "Cobertura meteorológica",
        "PASS",
        "366 fechas con correspondencia NOAA"
    ),
    (
        "Consistencia meteorológica",
        "PASS",
        f"{weather_audit['dates_checked']} fechas verificadas"
    ),
    (
        "Indicadores de calidad",
        "PASS",
        f"{len(QUALITY_FLAGS)} indicadores conservados"
    ),
    (
        "Valores faltantes de viento",
        "PASS",
        f"{total_missing_awnd:,} viajes identificados"
    )
]

df_validation_report = spark.createDataFrame(
    report,
    ["control", "status", "evidence"]
)

df_validation_report.show(truncate=False)

assert all(
    row["status"] == "PASS"
    for row in df_validation_report.collect()
)

print("Reporte integral de validación completado.")

+---------------------------+------+----------------------------------------+
|control                    |status|evidence                                |
+---------------------------+------+----------------------------------------+
|Integridad estructural     |PASS  |Esquemas y columnas requeridas correctos|
|Conciliación mensual       |PASS  |12 meses sin diferencias de registros   |
|Conservación de viajes     |PASS  |41,169,664 viajes conservados           |
|Cobertura meteorológica    |PASS  |366 fechas con correspondencia NOAA     |
|Consistencia meteorológica |PASS  |366 fechas verificadas                  |
|Indicadores de calidad     |PASS  |5 indicadores conservados               |
|Valores faltantes de viento|PASS  |428,126 viajes identificados            |
+---------------------------+------+----------------------------------------+

Reporte integral de validación completado.


### Lectura del reporte

**7 de 7 controles — PASS.** No se detectaron diferencias en los volúmenes mensuales ni en las cinco variables meteorológicas auditadas. El resultado corresponde a la versión de los archivos Silver analizada en esta ejecución.

La tabla se muestra en el notebook, pero **no se exporta a un archivo de auditoría** en esta etapa. Para un entorno productivo, convendría persistir un reporte con identificador de ejecución, fecha, versiones de datos y métricas verificables.


## 8. Conclusiones y limitaciones

La validación integral permitió comprobar la **consistencia estructural, temporal y analítica** de los tres datasets Silver generados en la fase 04.

### Principales resultados

- Se conservaron **41,169,664** registros de NYC Taxi Silver después de integrarlos con NOAA.
- Las diferencias de registros fueron **cero en los doce meses de 2024**.
- Se verificó la correspondencia meteorológica de **366 fechas**, sin inconsistencias en las cinco variables principales.
- Los **cinco indicadores de calidad** evaluados conservaron sus distribuciones mensuales.
- Se identificaron **428,126 viajes** asociados a fechas sin velocidad media del viento, sin realizar imputaciones artificiales.
- Los **siete controles** del reporte integral obtuvieron resultados satisfactorios.

### Limitaciones de la validación

1. **Trazabilidad individual.** La igualdad de volúmenes y conteos agregados no garantiza la identidad fila por fila. Para ampliar la auditoría sería necesario definir identificadores estables o huellas reproducibles de los registros.
2. **Alcance de las capas.** Se comparan los datasets Silver ya generados. No se recalculan en este notebook todos los criterios Raw → Silver ni se certifica la veracidad de los registros originales.
3. **Representatividad meteorológica.** NOAA corresponde a una única estación de Central Park, con resolución diaria. Sus mediciones pueden no representar el clima preciso de cada zona u hora de viaje.
4. **Uso predictivo.** Para pronósticos intradía, utilizar variables meteorológicas realizadas al cierre del mismo día podría introducir información futura; habría que utilizar datos disponibles en el instante de predicción.

### Conclusión general

El pipeline ETL **cumple los controles definidos para esta fase**. El dataset integrado está disponible en Parquet para análisis posteriores, con las anomalías y valores faltantes identificados y documentados. Los resultados permiten cerrar la fase 04 desde el punto de vista de integridad de las capas Silver, sin confundir esa aprobación con una certificación absoluta de calidad.
